In [0]:
from pyspark.sql.functions import (col, concat_ws,lit,to_date,trim, upper, lower)

In [0]:
%run "../includes/common_functions"

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS f1.silver;

In [0]:
drivers_df = spark.read.table("f1.bronze.drivers")

In [0]:
drivers_renamed_df = drivers_df \
    .withColumnRenamed("driverId", "driver_id")\
    .withColumnRenamed("driverRef","driver_ref")\
    .withColumnRenamed("dob", "date_of_birth")

In [0]:
drivers_transformed_name_df = drivers_renamed_df\
    .withColumn("name", concat_ws(" ", col("forename"), col("surname")))

In [0]:
drivers_clean_df = drivers_transformed_name_df\
    .filter(col("driver_id").isNotNull())\
    .dropDuplicates(["driver_id"])

In [0]:
drivers_final_df = add_ingestion_date(drivers_clean_df) \
    .withColumn("environment", lit("production")) \
    .withColumn("file_date", to_date(lit("2026-05-07")))

In [0]:
drivers_final_df = drivers_final_df \
    .withColumn("driver_ref", lower(trim(col("driver_ref")))) \
    .withColumn("code", trim(col("code"))) \
    .withColumn("forename", trim(col("forename"))) \
    .withColumn("surname", trim(col("surname"))) \
    .withColumn("nationality", trim(col("nationality"))) \
    .withColumn("url", trim(col("url")))

In [0]:
drivers_final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("f1.silver.drivers")

In [0]:
%sql
SELECT * FROM f1.silver.drivers;